# transformer-from-scratch: the checks that don't fit in a browser

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/superkush06/transformer-from-scratch/blob/main/notebooks/audit.ipynb)

The [browser demo](https://superkush06.github.io/transformer-from-scratch/demo/)
runs this package under WebAssembly, which puts three limits on it: it's
roughly 2.5x slower than native, it can't import anything outside numpy, and
I'm not going to run your code in it.

This notebook is the three checks those limits rule out.

1. **PyTorch as a second opinion.** Every gradient in this repo is checked
   against a finite difference, and a finite difference shares the forward
   pass with the analytic gradient. If my forward pass were wrong, both sides
   would move together and agree anyway. Autograd is an independent
   derivation, so it catches that.
2. **The audit at a size the browser won't do.** The demo samples above
   5,000 scalars because the cost is quadratic. Here it doesn't have to.
3. **Break a gradient yourself.** The demo gives you three preset bugs
   because running your code in the page is a bad idea. In a notebook,
   editing the cell is the whole interface.


In [ ]:
!pip install -q git+https://github.com/superkush06/transformer-from-scratch.git

import math

import numpy as np

from tfs import GPT, AdamLite
from tfs.ops import softmax_crossentropy

print('ready')


## 1. Against PyTorch

The same architecture rebuilt with `torch` ops, my weights copied in, and the
gradients diffed. Nothing is shared between the two paths except the numbers
in the weight matrices.

Both run in float64 so the comparison isn't limited by precision.


In [ ]:
import torch
import torch.nn.functional as F


def mirror(m, n_heads, ids, tgt):
    """The same forward pass, in torch, from the same weights."""
    named = dict(m.named_params())
    key = {'token_emb':'token_emb','pos_emb':'pos_emb','ln_f_g':'ln_f_g',
           'ln_f_b':'ln_f_b','lm_W':'lm_head.W'}
    for b in range(len(m.blocks)):
        key.update({f'b{b}.Wq':f'blocks.{b}.attn.W_q', f'b{b}.Wk':f'blocks.{b}.attn.W_k',
                    f'b{b}.Wv':f'blocks.{b}.attn.W_v', f'b{b}.Wo':f'blocks.{b}.attn.W_o',
                    f'b{b}.ln1_g':f'blocks.{b}.ln1_g', f'b{b}.ln1_b':f'blocks.{b}.ln1_b',
                    f'b{b}.ln2_g':f'blocks.{b}.ln2_g', f'b{b}.ln2_b':f'blocks.{b}.ln2_b',
                    f'b{b}.up_W':f'blocks.{b}.ffn.up.W', f'b{b}.up_b':f'blocks.{b}.ffn.up.b',
                    f'b{b}.dn_W':f'blocks.{b}.ffn.down.W', f'b{b}.dn_b':f'blocks.{b}.ffn.down.b'})
    P = {k: torch.tensor(named[v].data, dtype=torch.float64, requires_grad=True)
         for k, v in key.items()}

    D, H = m.d_model, n_heads
    dh, T = D // H, ids.shape[1]
    it, tt = torch.tensor(ids), torch.tensor(tgt)
    x = P['token_emb'][it] + P['pos_emb'][:T].unsqueeze(0)
    causal = torch.triu(torch.ones(T, T, dtype=torch.bool), 1)
    for b in range(len(m.blocks)):
        h1 = F.layer_norm(x, (D,), P[f'b{b}.ln1_g'], P[f'b{b}.ln1_b'], 1e-5)
        q = (h1 @ P[f'b{b}.Wq']).view(-1, T, H, dh).transpose(1, 2)
        k = (h1 @ P[f'b{b}.Wk']).view(-1, T, H, dh).transpose(1, 2)
        v = (h1 @ P[f'b{b}.Wv']).view(-1, T, H, dh).transpose(1, 2)
        s = (q @ k.transpose(-1, -2) / math.sqrt(dh)).masked_fill(causal, -1e9)
        merged = (torch.softmax(s, -1) @ v).transpose(1, 2).reshape(-1, T, D)
        x = x + merged @ P[f'b{b}.Wo']
        h2 = F.layer_norm(x, (D,), P[f'b{b}.ln2_g'], P[f'b{b}.ln2_b'], 1e-5)
        up = F.gelu(h2 @ P[f'b{b}.up_W'] + P[f'b{b}.up_b'], approximate='tanh')
        x = x + up @ P[f'b{b}.dn_W'] + P[f'b{b}.dn_b']
    h = F.layer_norm(x, (D,), P['ln_f_g'], P['ln_f_b'], 1e-5)
    logits = h @ P['lm_W']
    loss = F.cross_entropy(logits.reshape(-1, logits.shape[-1]), tt.reshape(-1))
    loss.backward()
    return float(loss.detach()), {v: P[k].grad.numpy() for k, v in key.items()}


IDS = np.array([[1,2,1,2,5],[3,3,0,1,1]])
TGT = np.array([[2,1,2,5,6],[3,0,1,1,4]])

print(f"{'architecture':>18}{'params':>9}{'my loss':>13}{'torch':>13}{'worst grad':>13}")
for d_model, n_heads, n_blocks, d_ff in [(8,2,2,16),(16,4,2,32),(24,3,3,48),(32,4,2,64)]:
    m = GPT(vocab_size=7, d_model=d_model, n_heads=n_heads, d_ff=d_ff,
            n_blocks=n_blocks, max_T=6, seed=0)
    for p in m.params():
        p.zero_grad()
    mine = m.loss_and_grads(IDS, TGT)
    theirs, grads = mirror(m, n_heads, IDS, TGT)
    worst = max(np.abs(p.grad - grads[n]).max() / max(np.abs(grads[n]).max(), 1e-12)
                for n, p in m.named_params())
    n_par = sum(p.data.size for p in m.params())
    cfg = f'{d_model}/{n_heads}/{n_blocks}/{d_ff}'
    print(f'{cfg:>18}{n_par:>9,}{mine:>13.8f}{theirs:>13.8f}{worst:>13.2e}')


Around `1e-15`, which is where float64 runs out of digits. The two
implementations aren't close, they're the same function.

This is a stronger statement than the finite-difference check in the repo,
which bottoms out near `3e-07` because the difference quotient has error of
its own. It's also a different kind of statement: `torch` was written by
other people from the same mathematics, so agreeing with it rules out a whole
class of mistake that checking against my own forward pass cannot.


## 2. The audit, without the browser's size limit

Every scalar in a model too big for the demo to sweep. The cost is two
forward passes per parameter and the forward pass grows with the width, so
this climbs fast. Start small if you're on a slow runtime.


In [ ]:
import time


def audit(m, ids, tgt, eps=1e-5):
    """Every parameter, analytic against central difference."""
    for p in m.params():
        p.zero_grad()
    m.loss_and_grads(ids, tgt)

    def loss():
        logits, _ = m.forward(ids)
        return float(softmax_crossentropy(logits, tgt)[0])

    worst, where, n = 0.0, '', 0
    for name, p in m.named_params():
        for c in range(p.data.size):
            a = float(p.grad.flat[c])
            o = float(p.data.flat[c])
            p.data.flat[c] = o + eps; up = loss()
            p.data.flat[c] = o - eps; dn = loss()
            p.data.flat[c] = o
            central = (up - dn) / (2 * eps)
            rel = abs(a - central) / max(abs(central), 1e-12)
            n += 1
            if rel > worst:
                worst, where = rel, f'{name}[{c}]'
    return n, worst, where


m = GPT(vocab_size=7, d_model=32, n_heads=4, d_ff=64, n_blocks=2, max_T=6, seed=0)
t0 = time.time()
n, worst, where = audit(m, IDS, TGT)
print(f'{n:,} derivatives in {time.time()-t0:.1f}s')
print(f'worst relative error {worst:.2e} at {where}')
print(f'CI tolerance is 1e-4, so that is {1e-4/worst:,.0f}x inside it')


Note how much less room there is at this size. On the 1,312-parameter model
the finite-difference check lands about 300x inside the tolerance CI enforces;
at 17,536 it's closer to 2x. That isn't the derivation getting worse, it's the
measurement getting worse: a wider model means more terms in every dot
product, so the two perturbed losses agree in more digits before they're
subtracted, and cancellation eats the answer.

Which is the argument for the first section. The `torch` comparison sat at
`1e-15` at every size, because it never subtracts two nearly equal numbers.
Finite differences tell you whether a derivative is roughly right; autograd
tells you whether it's the same function.


## 3. Break one yourself

Below is `layernorm_backward` exactly as it ships. The last term exists
because `x_hat` depends on the variance, which depends on every coordinate.

Delete it, or the one before it, or change a sign, then run the cell after
this one. The demo offers three preset mistakes; here you can make your own.


In [ ]:
import tfs.layers as layers
import tfs.ops as ops


def my_layernorm_backward(d_out, cache):
    x_hat, gamma, inv = cache
    N = x_hat.shape[-1]
    d_gamma = (d_out * x_hat).sum(axis=tuple(range(d_out.ndim - 1)))
    d_beta  = d_out.sum(axis=tuple(range(d_out.ndim - 1)))
    d_x_hat = d_out * gamma
    d_x = (1.0 / N) * inv * (
        N * d_x_hat
        - d_x_hat.sum(axis=-1, keepdims=True)
        - x_hat * (d_x_hat * x_hat).sum(axis=-1, keepdims=True)   # <-- try deleting this
    )
    return d_x, d_gamma, d_beta


# layers.py did `from .ops import ...`, so the name is bound in both modules
# and both have to move or the swap only takes in half the network
for mod in (ops, layers):
    if hasattr(mod, 'layernorm_backward'):
        mod.layernorm_backward = my_layernorm_backward

m = GPT(vocab_size=7, d_model=8, n_heads=2, d_ff=16, n_blocks=2, max_T=6, seed=0)
n, worst, where = audit(m, IDS, TGT)
print(f'{n:,} derivatives, worst relative error {worst:.2e} at {where}')
print('VERDICT:', 'agrees' if worst < 1e-4 else f'WRONG, by a factor of {worst:,.0f}')


## 4. Train it, and watch the loss

The period-5 task the demo uses. Correct gradients are necessary for this to
work and nowhere near sufficient, which is the point of the last cell.


In [ ]:
import matplotlib.pyplot as plt

for mod in (ops, layers):          # put the real one back first
    if hasattr(mod, 'layernorm_backward'):
        mod.layernorm_backward = ops.layernorm_backward

PERIOD = [1, 2, 3, 4, 5]
rng = np.random.default_rng(0)
m = GPT(vocab_size=7, d_model=8, n_heads=2, d_ff=16, n_blocks=2, max_T=6, seed=0)
opt = AdamLite(m.params(), lr=5e-3)

losses = []
for step in range(400):
    ids = np.zeros((8, 6), dtype=int); tgt = np.zeros((8, 6), dtype=int)
    for b in range(8):
        off = int(rng.integers(len(PERIOD)))
        seq = [PERIOD[(off + i) % len(PERIOD)] for i in range(7)]
        ids[b], tgt[b] = seq[:6], seq[1:]
    opt.zero_grad()
    losses.append(m.loss_and_grads(ids, tgt))
    opt.step()

plt.figure(figsize=(7, 3.2))
plt.semilogy(losses, lw=1.4)
plt.xlabel('step'); plt.ylabel('cross-entropy'); plt.title('400 steps, 1,312 parameters')
plt.grid(alpha=.25); plt.tight_layout(); plt.show()

print('prompt 1 2 3 ->', m.generate(np.array([1,2,3]), max_new=7, temperature=0.0)[3:])


---

The engine is [transformer-from-scratch](https://github.com/superkush06/transformer-from-scratch).
The interactive version of experiments 2 and 3 is
[here](https://superkush06.github.io/transformer-from-scratch/demo/), running
the same package compiled to WebAssembly.
